# Exercise 4 – Example: scene prompt → clip → consistency

This notebook shows how to plan a video clip and measure whether a character keeps its look, with as little code as possible. **Run every cell and read the code.** Then open `video_generation_tasks.ipynb`: there you plan and check a new scene.

1. **Plan:** let the OpenAI API write the prompt for one scene.
2. **Generate:** the clip is made by hand in Google Flow, with a character sheet as reference.
3. **Check:** compare the character in the clip with the sheet, frame by frame.

## Setup

Select the **Python 3.11** kernel. The OpenAI API key comes from your `OPENAI_API_KEY` Codespaces secret. This notebook uses the **OpenAI API** (model `gpt-6-luna`); other providers, e.g. Google Gemini or Anthropic Claude, have their own APIs with different code.

In [ ]:
# Python libraries.
import sys
from typing import Literal

import pandas as pd
from pydantic import BaseModel

# helper.py is in the main folder, one level up.
sys.path.append("..")
from helper import DATA, ask, extract_frames, show_images

## 1. Plan: the prompt for one scene

The model writes the five parts of a video prompt. The character descriptions are added **word for word** in front, so they never change.

In [ ]:
CHARACTER = "Ueli, red beanie, olive jacket"
DOG = "Bruno, Bernese, red collar"


class ScenePrompt(BaseModel):
    shot: str
    camera: str
    action: str
    light: str
    sound: str


PROMPT = f"""
Write one short video prompt for Google Flow.
The scene: {CHARACTER} together with the dog {DOG} on the shore of a mountain lake.
Describe shot, camera, action, light and sound, each as one short sentence.
"""

plan = ask(PROMPT, ScenePrompt, [])  # the empty list means: no images
flow_prompt = f"{CHARACTER}. {DOG}. " + " ".join(plan.values())
print(flow_prompt)

## 2. Generate: the character sheet and Google Flow

The character sheet is the reference picture. In Google Flow (by hand, not in this notebook) it is added as **ingredient**, together with the prompt above. Ueli's clip from Exercise 2 was made this way.

In [ ]:
sheet = DATA / "reference" / "ueli_sheet.jpg"
show_images([sheet])

## 3. Check one feature, frame by frame

One frame per second is compared with the sheet. The answer is one of three ratings: `match`, `differs` or `not visible`.

In [ ]:
paths = extract_frames(DATA / "videos" / "ueli_scene3_talk.mp4")
show_images(paths)

In [ ]:
Rating = Literal["match", "differs", "not visible"]


class Beanie(BaseModel):
    beanie: Rating


PROMPT = """
Image 1 is the reference. Rate the beanie in image 2.
Use match for a visible match, differs for a visible change
and not visible when it is unclear or out of frame.
"""

rows = []
for path in paths:
    row = ask(PROMPT, Beanie, [sheet, path])  # the sheet comes first
    row["frame"] = path.name
    rows.append(row)

ratings = pd.DataFrame(rows)
print((ratings["beanie"] == "match").sum(), "of", len(ratings), "frames match")
ratings

**Next:** open `video_generation_tasks.ipynb` and plan and check your own scene.

### Jupyter notebook --footer info--

In [ ]:
# Print system and version info for this run.
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')